# Ropedia Academy — A1 · Parametric body models (SMPL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A1.ipynb)

> **Builds the full SMPL pipeline — shape blendshapes → pose correctives → linear blend skinning — and shows ~80 differentiable numbers produce a valid mesh.**
>
> 搭出完整的 SMPL 流程——形状混合形变 → 姿态校正 → 线性混合蒙皮——并展示约 80 个可微数字即可生成一具有效网格。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A1

In [ ]:
import torch

# Miniature SMPL: M(beta, theta) -> a full mesh from ~80 numbers.
# (Real SMPL: 6890 verts, 10 shape coeffs + 24 joint rotations. We shrink to run.)
torch.manual_seed(0)
V, n_beta, J = 60, 10, 5
v_template = torch.randn(V, 3)                  # neutral template body
shape_dirs = torch.randn(V, 3, n_beta)         # shape blendshapes (identity)
pose_dirs  = torch.randn(V, 3, J * 9)          # pose-corrective blendshapes
weights    = torch.softmax(torch.randn(V, J), dim=1)   # LBS skinning weights

def aa_to_R(aa):                               # axis-angle (J,3) -> rotations (J,3,3)
    th = aa.norm(dim=-1, keepdim=True) + 1e-8; k = aa / th
    Kx = torch.zeros(aa.shape[0], 3, 3)
    Kx[:,0,1], Kx[:,0,2], Kx[:,1,0] = -k[:,2], k[:,1], k[:,2]
    Kx[:,1,2], Kx[:,2,0], Kx[:,2,1] = -k[:,0], -k[:,1], k[:,0]
    I = torch.eye(3).expand_as(Kx)
    return I + th.sin()[...,None]*Kx + (1-th.cos())[...,None]*(Kx @ Kx)

def smpl(beta, theta):                          # theta: (J,3) joint rotations
    R = aa_to_R(theta)
    v = v_template + shape_dirs @ beta          # 1) shape blendshapes
    v = v + pose_dirs @ (R - torch.eye(3)).reshape(-1)   # 2) pose correctives
    return torch.einsum('vj,jab,vb->va', weights, R, v)  # 3) linear blend skinning

beta  = torch.randn(n_beta, requires_grad=True)
theta = torch.randn(J, 3, requires_grad=True)
mesh  = smpl(beta, theta)
print("params:", beta.numel() + theta.numel(), "-> mesh vertices:", tuple(mesh.shape))
mesh.sum().backward()                            # every body is differentiable in (beta, theta)
print("differentiable in shape & pose:", beta.grad is not None)

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A1
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks